# Aurora Inference + Rollout (Fine-Tuned Model)

This notebook provides inference/rollout using a **fine-tuned Aurora checkpoint**.

Workflow:
1. Load YAML config
2. Load test dataset
3. Initialize model with fine-tuned checkpoint
4. Run rollout predictions
5. Save predictions and plots

## Environment / Imports

In [1]:
# Setup: Add project root to Python path
import os
import sys
from pathlib import Path

os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')

# Get the project root (parent of finetune directory)
notebook_dir = Path.cwd()
if notebook_dir.name == 'finetune':
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root: {project_root}')

Project root: /data/aurora


In [2]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

# Plotting temporarily disabled; import matplotlib.pyplot as plt when re-enabled.
import numpy as np
import torch
import xarray as xr

import finetune.aurora_finetune_utils as ft

/home/azureuser/miniforge3/envs/aurora/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [3]:
# ============================================================================
# CONFIGURATION: Edit these settings (papermill parameters cell)
# ============================================================================

# Configuration file (auto-detects location)
CONFIG_PATH_NAME = 'aurora_NO2_finetune_US-WEST_3day_lead_flow_matching_transformer_config.yaml'
# Path to your fine-tuned checkpoint and output dir.
# Defaults derive from the YAML's `case_name` so artifacts land under
# outputs/<case_name>/ and outputs/checkpoints/<case_name>/, matching
# what aurora_finetune_distributed.py wrote during training. Set these
# explicitly to override.
# Use 'best' or 'last' from the YAML's case-specific checkpoint directory,
# an explicit checkpoint path, or None for automatic validated selection.
CHECKPOINT_PATH = None

# Rollout configuration (can override config YAML).
# For the regional NO2 and global O3 configs, 6 steps at 12 hours/step is 3 days.
# None = use config value, then max(data.target_lead_times) fallback.
ROLLOUT_NUM_STEPS = None

# Optional overrides; None reads refinement.ensemble_size / refinement.seed.
NUM_ENSEMBLE = None
ENSEMBLE_SEED = None
# Override the flow-refine sampling steps at inference time, independent of
# what training saved in the checkpoint. None = use the checkpoint's value.
# Set > 1 to force multi-step stochastic Flow-Matching sampling (required for
# ensemble spread) even if the checkpoint was saved as deterministic (1-step).
SAMPLING_STEPS_OVERRIDE = None
# A temporal checkpoint must use its saved sampler contract. Set this only
# for a deliberately unsafe diagnostic that changes Mamba's input distribution.
ALLOW_UNSAFE_MAMBA_SAMPLING_OVERRIDE = False
# 'on' is deployable causal inference; 'off' and 'shuffled' are ablation controls.
TEMPORAL_CONTROL = 'on'

# Output settings
SAVE_NETCDF = True
# SAVE_PLOTS = True  # Plot generation is temporarily disabled.
SAVE_PLOTS = False
# None = outputs/<case_name>/.
OUTPUT_DIR = None
OVERWRITE_EXISTING = True

# ============================================================================


## Load Test Dataset

In [4]:
# Resolve derived settings here, after papermill injects parameter overrides.
if Path(CONFIG_PATH_NAME).exists():
    CONFIG_PATH = Path(CONFIG_PATH_NAME)
elif (Path('finetune') / CONFIG_PATH_NAME).exists():
    CONFIG_PATH = Path('finetune') / CONFIG_PATH_NAME
else:
    raise FileNotFoundError(f'Config file {CONFIG_PATH_NAME} not found')

cfg = ft.load_config(CONFIG_PATH)
_case = str(cfg.get('case_name', '')).strip()
if isinstance(CHECKPOINT_PATH, str) and CHECKPOINT_PATH.lower() in {'best', 'last'}:
    CHECKPOINT_PATH = str(
        Path(cfg['paths']['checkpoint_dir']) / f'{CHECKPOINT_PATH.lower()}.ckpt'
    )
elif CHECKPOINT_PATH is None:
    CHECKPOINT_PATH = str(ft.select_refinement_checkpoint(
        cfg['paths']['checkpoint_dir'],
        require_validated=bool(cfg.get('inference', {}).get('require_validated_checkpoint', False)),
    ))
if OUTPUT_DIR is None:
    OUTPUT_DIR = cfg['paths']['output_dir']
print(f"Loaded config from: {CONFIG_PATH.resolve()}")
print(f"Case name: {_case}")
print(f"Test data: {cfg['paths']['test_data_path']}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

test_ds = ft.open_dataset(cfg['paths']['test_data_path'], cfg)

# Merge static variables from external pickle (lsm, z, slt) into the test dataset.
static_path = cfg['paths'].get('static_data_path', '')
if static_path:
    test_ds = ft.merge_external_static_vars(test_ds, static_path, cfg)
    print(f'Merged static vars from: {static_path}')

resolved_specs = ft.resolve_variable_specs(test_ds, cfg)
lon_periodic = ft.validate_longitude_consistency([test_ds], cfg)
lon_dim = str(cfg.get('data', {}).get('lon_dim', 'longitude'))
model_longitude = test_ds[lon_dim].values

print('Test dataset sizes:', test_ds.sizes)
print('Predictors:', [f"{s.dataset_name}->{s.aurora_name} ({s.kind})" for s in resolved_specs.predictors])
print('Targets:   ', [f"{s.dataset_name}->{s.aurora_name} ({s.kind})" for s in resolved_specs.targets])
print('Static:    ', [f"{s.dataset_name}->{s.aurora_name} ({s.kind})" for s in resolved_specs.static])

Loaded config from: /data/aurora/finetune/aurora_NO2_finetune_US-WEST_3day_lead_flow_matching_transformer_config.yaml
Case name: NO2_US-WEST_3day_lead_flow_matching_transformer
Test data: /data/aurora/data/NO2_US-WEST_3day_lead/test.nc
Checkpoint: /data/aurora/finetune/outputs/checkpoints/NO2_US-WEST_3day_lead_flow_matching_transformer/best.ckpt
Output dir: /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching_transformer
Merged static vars from: /data/cams/aurora-0.4-air-pollution-static.pickle
Test dataset sizes: Frozen({'time': 185, 'latitude': 53, 'longitude': 70, 'level': 13})
Predictors: ['t2m->2t (surf)', 'u10->10u (surf)', 'v10->10v (surf)', 'msl->msl (surf)', 'tcno2->tcno2 (surf)', 'z->z (atmos)', 'u->u (atmos)', 'v->v (atmos)', 't->t (atmos)', 'q->q (atmos)', 'no2->no2 (atmos)']
Targets:    ['no2->no2 (atmos)', 'tcno2->tcno2 (surf)']
Static:     ['lsm->lsm (static)', 'z_static->z (static)', 'slt->slt (static)', 'static_ammonia->static_ammonia (static)', 'static_am

## Build Test Samples

In [5]:
# For rollout inference we only need input_time_steps history frames to seed the model.
# Unlike training, there is no target lead-time tail constraint, so every anchor
# with enough input history is a valid forecast initialization time.
time_dim = cfg.get('data', {}).get('time_dim', 'time')
input_steps = int(cfg.get('data', {}).get('input_time_steps', 2))
n_time = int(test_ds.sizes[time_dim])

rollout_start_samples = [
    {
        'sample_index': sample_idx,
        'anchor_index': anchor_idx,
        'history_indices': list(range(anchor_idx - input_steps + 1, anchor_idx + 1)),
        'target_indices': {},
    }
    for sample_idx, anchor_idx in enumerate(range(input_steps - 1, n_time))
]
print(f'Number of rollout initialization times: {len(rollout_start_samples)}')

if not rollout_start_samples:
    raise ValueError('No valid start positions in test dataset (need at least input_time_steps timesteps).')

first_anchor_time = np.datetime64(
    test_ds[time_dim].values[rollout_start_samples[0]['anchor_index']], 's'
)
last_anchor_time = np.datetime64(
    test_ds[time_dim].values[rollout_start_samples[-1]['anchor_index']], 's'
)
print(f'First initialization time: {first_anchor_time}')
print(f'Last initialization time:  {last_anchor_time}')
print('Example first history indices:', rollout_start_samples[0]['history_indices'])


Number of rollout initialization times: 184
First initialization time: 2024-07-01T00:00:00
Last initialization time:  2024-09-30T12:00:00
Example first history indices: [0, 1]


## Initialize Model

In [6]:
# The checkpoint cell calls the shared factory after loading checkpoint
# normalization metadata, so training and inference cannot diverge architecturally.
model_cfg = cfg['model']
print('Model construction deferred to ft.load_model_from_checkpoint().')

Model construction deferred to ft.load_model_from_checkpoint().


## Load Fine-Tuned Checkpoint

In [7]:
import subprocess as _sp

checkpoint_path = Path(CHECKPOINT_PATH).expanduser()

def _pick_best_gpu():
    try:
        out = _sp.check_output(
            ['nvidia-smi', '--query-gpu=index,memory.free',
             '--format=csv,noheader,nounits'], text=True)
    except Exception:
        return 0
    best_idx, best_free = 0, 0
    for line in out.strip().splitlines():
        idx, free = (float(value) for value in line.split(','))
        if free > best_free:
            best_idx, best_free = int(idx), free
    return best_idx

device = torch.device(f'cuda:{_pick_best_gpu()}') if torch.cuda.is_available() else torch.device('cpu')
lat_dim = str(cfg.get('data', {}).get('lat_dim', 'latitude'))
model, checkpoint = ft.load_model_from_checkpoint(
    cfg,
    resolved_specs,
    checkpoint_path,
    lon=model_longitude,
    lat=test_ds[lat_dim].values,
    map_location='cpu',
    mmap=True,
    autocast=False,
    flow_sampling_steps_override=SAMPLING_STEPS_OVERRIDE,
    allow_unsafe_temporal_sampling_override=ALLOW_UNSAFE_MAMBA_SAMPLING_OVERRIDE,
)

from finetune.flow_refine import AuroraFlowRefine as _AFR
from finetune.refinement.two_phase import resolve_temporal_config

def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

checkpoint_sha256 = _sha256_file(checkpoint_path)
resolved_temporal_config = resolve_temporal_config(cfg)
if isinstance(model, _AFR):
    _contract_version = int(getattr(model, 'flow_refine_contract_version', 1))
    _feedback = (
        True if _contract_version == 1
        else bool(cfg.get('training', {}).get('flow_refine_autoregressive_feedback', False))
    )
    trajectory_policy = {
        'source': f'legacy_flow_contract_v{_contract_version}',
        'aurora_autoregressive_feedback': _feedback,
    }
else:
    _refinement_config = getattr(model, 'refinement_config', None)
    trajectory_policy = {
        'source': 'unified_refinement',
        'aurora_autoregressive_feedback': bool(
            getattr(_refinement_config, 'feedback_to_rollout', False)
        ),
    }

model = model.to(device)
model.eval()
print(f'Loaded checkpoint from: {checkpoint_path}')
print(f'  Refinement type: {checkpoint.get("refinement_type", "none")}')
print(f'  Epoch: {checkpoint.get("epoch", "N/A")}')
print(f'  Best val loss: {checkpoint.get("best_val_loss", "N/A")}')
print(f'Device: {device}')

Loaded checkpoint from: /data/aurora/finetune/outputs/checkpoints/NO2_US-WEST_3day_lead_flow_matching_transformer/best.ckpt
  Refinement type: flow_matching_transformer
  Epoch: 29
  Best val loss: 0.6565577056076813
Device: cuda:0


## Run Rollout Inference

In [8]:
from finetune.longitude import dateline_discontinuity_from_edges

output_dir = Path(OUTPUT_DIR).expanduser()
output_dir.mkdir(parents=True, exist_ok=True)

def _timestamp_tag(value):
    return str(np.datetime64(value, 's')).replace('-', '').replace(':', '')

# Override rollout steps if specified.
if ROLLOUT_NUM_STEPS is not None:
    cfg['rollout']['rollout_num_steps'] = ROLLOUT_NUM_STEPS

rollout_steps = int(cfg['rollout'].get('rollout_num_steps', 0))
if rollout_steps <= 0:
    # Fall back to max(target_lead_times) from data config. For the O3/NO2 US-WEST
    # 3-day configs this is 6 steps, and rollout_step_hours is 12.
    lead_times = cfg.get('data', {}).get('target_lead_times', [])
    if lead_times:
        rollout_steps = max(int(x) for x in lead_times)
        cfg['rollout']['rollout_num_steps'] = rollout_steps
    else:
        raise ValueError('data.target_lead_times must contain at least one lead.')


rollout_step_hours = int(cfg['rollout']['rollout_step_hours'])
forecast_hours = rollout_steps * rollout_step_hours
smooth_sigma = float(cfg.get('rollout', {}).get('smooth_sigma', 0.0))
patch_size = int(cfg.get('model', {}).get('patch_size', 3))

refinement_cfg = cfg.get('model', {}).get('refinement', {})
n_ensemble = max(1, int(refinement_cfg.get('ensemble_size', 1) if NUM_ENSEMBLE is None else NUM_ENSEMBLE))
ensemble_seed = int(refinement_cfg.get('seed', 0) if ENSEMBLE_SEED is None else ENSEMBLE_SEED)


# Legacy one-step flow correction is deterministic; unified diffusion/flow
# refiners draw independent members through refinement_seed above.
from finetune.flow_refine import AuroraFlowRefine as _AFR
if n_ensemble > 1 and isinstance(model, _AFR) and int(model.sampling_steps) <= 1:
    print(
        f'WARNING: NUM_ENSEMBLE={n_ensemble} but legacy flow sampling_steps=1; '
        'members will be identical.'
    )

from html import escape
from IPython.display import HTML, display


class _SingleDisplayProgress:
    """A notebook progress bar that updates one display instead of printing lines."""

    def __init__(self, iterable, desc='', unit='', **_):
        self.iterable = iterable
        self.total = len(iterable)
        self.desc = desc
        self.unit = unit
        self.n = 0
        self.postfix = {}
        self.display_handle = None

    def _html(self):
        maximum = max(self.total, 1)
        percent = 100.0 * self.n / maximum
        label = f'{self.desc}: {self.n}/{self.total} {self.unit} ({percent:.0f}%)'
        details = ' | '.join(f'{key}={value}' for key, value in self.postfix.items())
        details_html = f'<div style="color:#666">{escape(details)}</div>' if details else ''
        return HTML(
            f'<div><div>{escape(label)}</div>'
            f'<progress value="{self.n}" max="{maximum}" style="width:100%"></progress>'
            f'{details_html}</div>'
        )

    def _refresh(self):
        if self.display_handle is not None:
            self.display_handle.update(self._html())

    def __enter__(self):
        self.display_handle = display(self._html(), display_id=True)
        return self

    def __exit__(self, exc_type, _exc_value, _traceback):
        if exc_type is not None:
            self.postfix['status'] = 'interrupted'
        self._refresh()
        return False

    def __iter__(self):
        for index, item in enumerate(self.iterable):
            self.n = index
            yield item
            self.n = index + 1
            self._refresh()

    def set_postfix(self, refresh=True, **values):
        self.postfix = values
        if refresh:
            self._refresh()


rollout_results = []
with _SingleDisplayProgress(
    rollout_start_samples,
    desc='Inference rollouts',
    unit='init',
    dynamic_ncols=True,
    leave=False,
    position=0,
    mininterval=1.0,
) as pbar:
    for rollout_idx, start_sample in enumerate(pbar):
        anchor_time = np.datetime64(test_ds[time_dim].values[start_sample['anchor_index']], 's')
        history_times = [
            np.datetime64(test_ds[time_dim].values[idx], 's')
            for idx in start_sample['history_indices']
        ]
        init_tag = _timestamp_tag(anchor_time)
        rollout_path = output_dir / f'rollout_predictions_init_{init_tag}.nc'
        if rollout_path.exists() and not OVERWRITE_EXISTING:
            raise FileExistsError(
                f'Refusing to overwrite existing rollout: {rollout_path}. '
                'Choose a new OUTPUT_DIR or explicitly set OVERWRITE_EXISTING=True.'
            )

        # Draw n_ensemble stochastic rollouts for this initialization. Each member
        # uses a distinct seed so Flow Matching produces independent samples.
        member_datasets = []
        member_seeds = []
        member_pred_counts = []
        for member_idx in range(n_ensemble):
            seed = ft.derive_refinement_seed(ensemble_seed, anchor_time, member_idx)
            member_seeds.append(int(seed))
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

            pbar.set_postfix(
                init=str(anchor_time),
                member=f'{member_idx + 1}/{n_ensemble}',
                status='running',
                refresh=True,
            )

            predictions = ft.run_rollout(
                model=model,
                ds=test_ds,
                start_sample=start_sample,
                config=cfg,
                resolved_specs=resolved_specs,
                device=device,
                refinement_seed=seed,
                refinement_ensemble_size=1,  # This loop writes individual members.
                temporal_control=TEMPORAL_CONTROL,
            )
            member_pred_counts.append(len(predictions))

            if predictions and SAVE_NETCDF:
                member_ds = ft.save_predictions(
                    predictions,
                    rollout_path,
                    save_netcdf=False,
                    resolved_specs=resolved_specs,
                    smooth_sigma=smooth_sigma,
                    patch_size=patch_size,
                    lon_periodic=lon_periodic,
                    initialization_time=anchor_time,
                )
                member_datasets.append(member_ds)

            del predictions
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        num_predictions = member_pred_counts[0] if member_pred_counts else 0
        result = {
            'rollout_idx': rollout_idx,
            'start_sample': start_sample,
            'anchor_time': anchor_time,
            'history_times': history_times,
            'num_predictions': num_predictions,
            'num_members': n_ensemble,
        }

        if member_datasets and SAVE_NETCDF:
            # Single member -> keep the flat (time, lat, lon) layout. Multiple
            # members -> stack along a new leading 'member' dimension.
            if len(member_datasets) == 1:
                rollout_ds = member_datasets[0]
            else:
                rollout_ds = xr.concat(member_datasets, dim='member', join='exact')
                rollout_ds = rollout_ds.assign_coords(member=np.arange(len(member_datasets)))

            rollout_ds.attrs['initialization_time'] = str(anchor_time)
            rollout_ds.attrs['anchor_index'] = int(start_sample['anchor_index'])
            rollout_ds.attrs['history_times'] = ','.join(str(t) for t in history_times)
            rollout_ds.attrs['num_ensemble_members'] = len(member_datasets)
            rollout_ds.attrs['correction_convention'] = refinement_cfg.get('correction_convention', 'cams_minus_aurora')
            rollout_ds.attrs['refined_forecast_equation'] = 'Aurora + predicted_correction_physical'
            rollout_ds.attrs['field_semantics'] = 'final_refined_physical_forecast'
            rollout_ds.attrs['checkpoint_path'] = str(Path(checkpoint_path).resolve())
            rollout_ds.attrs['checkpoint_sha256'] = checkpoint_sha256
            rollout_ds.attrs['checkpoint_epoch'] = int(checkpoint.get('epoch', -1))
            rollout_ds.attrs['resolved_flow_sampling_steps'] = int(
                checkpoint.get('resolved_flow_sampling_steps', getattr(model, 'sampling_steps', 0))
            )
            rollout_ds.attrs['flow_sampling_steps_source'] = str(
                checkpoint.get('flow_sampling_steps_source', 'not_applicable')
            )
            rollout_ds.attrs['temporal_enabled'] = int(bool(getattr(model, 'has_temporal', False)))
            rollout_ds.attrs['temporal_control'] = str(TEMPORAL_CONTROL)
            rollout_ds.attrs['resolved_temporal_config'] = json.dumps(
                resolved_temporal_config, sort_keys=True
            )
            rollout_ds.attrs['temporal_scan_backend'] = str(
                resolved_temporal_config['scan_backend']
            )
            rollout_ds.attrs['temporal_semantic_version'] = int(
                resolved_temporal_config['semantic_version']
            )
            rollout_ds.attrs['trajectory_policy'] = json.dumps(
                trajectory_policy, sort_keys=True
            )
            rollout_ds.attrs['base_ensemble_seed'] = int(ensemble_seed)
            rollout_ds.attrs['member_seeds'] = ','.join(str(seed) for seed in member_seeds)
            rollout_ds.attrs['forecast_product'] = (
                'deterministic_refined_forecast' if refinement_cfg.get('deterministic_inference', False)
                else 'stochastic_refined_members'
            )
            if lon_periodic:
                for _name, _da in rollout_ds.data_vars.items():
                    _edge = [_da.isel(longitude=_i).values for _i in (0, 1, -2, -1)]
                    _seam = dateline_discontinuity_from_edges(*_edge)
                    print(
                        f'{_name} dateline diagnostic: jump={_seam["seam_jump"]:.4g}, '
                        f'local_ratio={_seam["local_ratio"]:.3f}'
                    )
            for _coord in ('time', 'latitude', 'longitude', 'level'):
                if _coord in rollout_ds.coords:
                    rollout_ds[_coord].encoding['_FillValue'] = None
            rollout_ds.to_netcdf(str(rollout_path))
            rollout_ds.close()
            for member_ds in member_datasets:
                member_ds.close()

            result['rollout_path'] = rollout_path
            pbar.set_postfix(
                init=str(anchor_time),
                steps=num_predictions,
                members=len(member_datasets),
                status='saved',
                refresh=True,
            )
        else:
            pbar.set_postfix(
                init=str(anchor_time),
                steps=num_predictions,
                members=n_ensemble,
                status='done',
                refresh=True,
            )

        rollout_results.append(result)

print(
    f'Completed {len(rollout_results)} initialization(s); '
    f'{rollout_steps} steps ({forecast_hours} forecast hours) each, '
    f'{n_ensemble} ensemble member(s) per initialization.'
)
if SAVE_NETCDF and smooth_sigma > 0:
    print(f'Gaussian smoothing: sigma={smooth_sigma}')


Completed 184 initialization(s); 6 steps (72 forecast hours) each, 1 ensemble member(s) per initialization.


## Save Predictions to NetCDF

In [9]:
# NetCDF files are saved inside the rollout loop above so each initialization
# can be written immediately without retaining all predictions in memory.


## Plot Results

In [10]:
# Result plotting is temporarily disabled. NetCDF outputs are still written above.
# Uncomment this cell and restore the plotting imports/flag when plots are needed.

# saved_rollout_results = [result for result in rollout_results if 'rollout_path' in result]
# if saved_rollout_results and SAVE_PLOTS:
#     plot_vars = cfg.get('notebook', {}).get('plot_variables', [])
#
#     for result in saved_rollout_results:
#         rollout_ds = xr.open_dataset(result['rollout_path'])
#         try:
#             if not plot_vars:
#                 # Default: first 3 dataset variable names from the current rollout.
#                 plot_vars_for_result = list(rollout_ds.data_vars.keys())[:3]
#             else:
#                 plot_vars_for_result = plot_vars
#
#             init_tag = _timestamp_tag(result['anchor_time'])
#             for var in plot_vars_for_result:
#                 if var not in rollout_ds:
#                     print(f'Variable {var} not in rollout dataset for {init_tag}, skipping')
#                     continue
#
#                 da = rollout_ds[var]
#                 # Collapse the ensemble dimension to its mean for plotting.
#                 if 'member' in da.dims:
#                     da = da.mean('member')
#                 has_level = 'level' in da.dims
#
#                 # Plot each time step, capped for readability.
#                 n_times = da.sizes.get('time', 1)
#                 num_steps = min(n_times, 4)
#                 fig, axes = plt.subplots(1, num_steps, figsize=(5 * num_steps, 5))
#                 if num_steps == 1:
#                     axes = [axes]
#
#                 for step_i, ax in enumerate(axes):
#                     if has_level:
#                         image = da.isel(time=step_i, level=-1)
#                         level_val = float(da['level'].values[-1])
#                         title = f'{var} | init={result["anchor_time"]} | t={step_i} | {level_val:.0f} hPa'
#                     else:
#                         image = da.isel(time=step_i)
#                         title = f'{var} | init={result["anchor_time"]} | t={step_i}'
#
#                     plot_lon, plot_values = add_cyclic_column(
#                         da['longitude'].values, image.values, only_if_periodic=True,
#                     )
#                     mesh = ax.pcolormesh(
#                         plot_lon, da['latitude'].values,
#                         plot_values, cmap='viridis', shading='auto',
#                     )
#                     fig.colorbar(mesh, ax=ax, shrink=0.8)
#                     ax.set_title(title)
#                     ax.set_xlabel('longitude')
#                     ax.set_ylabel('latitude')
#
#                 fig.tight_layout()
#                 fig_path = output_dir / f'rollout_init_{init_tag}_{var}.png'
#                 fig.savefig(fig_path, dpi=150, bbox_inches='tight')
#                 print(f'Saved plot: {fig_path}')
#                 plt.close(fig)
#         finally:
#             rollout_ds.close()
# else:
#     print('Skipping plots (SAVE_PLOTS=False or no saved rollout data)')


## Summary

In [11]:
print('\n' + '='*70)
print('INFERENCE SUMMARY')
print('='*70)
print(f'Checkpoint: {checkpoint_path}')
print(f'Initialization count: {len(rollout_results)}')
print(f'Rollout steps per initialization: {rollout_steps}')
print(f'Ensemble members per initialization: {n_ensemble}')
print(f'Forecast hours per initialization: {forecast_hours}')
print(f'Output directory: {output_dir}')
saved_paths = [str(result['rollout_path']) for result in rollout_results if 'rollout_path' in result]
print(f'NetCDF files saved: {len(saved_paths)}')
for path in saved_paths[:5]:
    print(f'  {path}')
if len(saved_paths) > 5:
    print(f'  ... {len(saved_paths) - 5} more')
print('='*70)



INFERENCE SUMMARY
Checkpoint: /data/aurora/finetune/outputs/checkpoints/NO2_US-WEST_3day_lead_flow_matching_transformer/best.ckpt
Initialization count: 184
Rollout steps per initialization: 6
Ensemble members per initialization: 1
Forecast hours per initialization: 72
Output directory: /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching_transformer
NetCDF files saved: 184
  /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching_transformer/rollout_predictions_init_20240701T000000.nc
  /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching_transformer/rollout_predictions_init_20240701T120000.nc
  /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching_transformer/rollout_predictions_init_20240702T000000.nc
  /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching_transformer/rollout_predictions_init_20240702T120000.nc
  /data/aurora/finetune/outputs/NO2_US-WEST_3day_lead_flow_matching_transformer/rollout_predictions_init_20240703T